# NLP2 - TP1 - TinyGPT

Referencias:

* [Consigna](https://github.com/FIUBA-Posgrado-Inteligencia-Artificial/CEIA-LLMIAG/blob/main/ClaseIV/TinyGPT_es.ipynb)
* [Util de training](https://github.com/FIUBA-Posgrado-Inteligencia-Artificial/CEIA-LLMIAG/blob/main/ClaseIV/trainer.py)

In [8]:
# imports
import httpx
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

# training
import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import StepLR

# source propio + util de training
from src.trainer import Trainer
from src.tinygpt import TinyGPT
from src.config import GPTConfig, MoEArgs
from src.mlp import MoEFFN
from src.generation import generate, SamplingParams

In [9]:
device =  'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'

device

'cuda'

In [10]:
def generate_n(n_generations: int, *args, **kwargs):
    for idx in range(n_generations):
        print(f"SAMPLE {idx+1:>2}/{n_generations:<2}")
        print(generate(*args, **kwargs))
        print("*"*20, end="\n\n")

In [11]:
def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## Data

In [12]:
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
response = httpx.get(url)
text = response.text

text = text[:100_000]  # solo 100k characters para speedup

# peek
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [13]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print("Using vocab size", vocab_size)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

def encode(s):
    return [stoi[c] for c in s]
def decode(l):
    return ''.join([itos[i] for i in l])

Using vocab size 61


In [14]:
# dense config
config = GPTConfig(vocab_size=vocab_size)

config

GPTConfig(vocab_size=61, block_size=32, batch_size=8, n_embd=64, n_head=4, n_layer=2, dropout=0.1, bias=True, ff_class=None, moe=None)

In [15]:
# 0.9-train/0.1-val split
data = torch.tensor(encode(text), dtype=torch.long)
split = int(0.9 * len(data))
train_data = data[:split]
val_data = data[split:]

train_data.shape, val_data.shape

(torch.Size([90000]), torch.Size([10000]))

In [16]:
# dataset
class CharDataset(Dataset):
    def __init__(self, data: torch.Tensor, block_size: int):
        self.data = data
        self.block_size = block_size

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.block_size]
        y = self.data[idx + 1 : idx + self.block_size + 1]
        return x, y

train_dataset = CharDataset(train_data, config.block_size)
val_dataset = CharDataset(val_data, config.block_size)

In [17]:
# dataloaders
num_workers = 0 if device=='mps' else 2

train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    drop_last=True,
    pin_memory=True,
    num_workers= num_workers,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    drop_last=True,
    pin_memory=True,
    num_workers=num_workers,
)

## Dense model

In [18]:
dense_m = TinyGPT(config).to(device)
dense_model = torch.compile(dense_m)
dense_model

OptimizedModule(
  (_orig_mod): TinyGPT(
    (token_emb): Embedding(61, 64)
    (pos_emb): Embedding(32, 64)
    (blocks): ModuleList(
      (0-1): 2 x Block(
        (ln1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (ln2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (attn): MultiHeadAttention(
          (heads): ModuleList(
            (0-3): 4 x AttentionHead(
              (key_query_value): Linear(in_features=64, out_features=48, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
          )
          (proj): Linear(in_features=64, out_features=64, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (ff): DenseFFN(
          (net): Sequential(
            (0): Linear(in_features=64, out_features=256, bias=True)
            (1): ReLU()
            (2): Linear(in_features=256, out_features=64, bias=True)
            (3): Dropout(p=0.1, inplace=False)
          )
        )
      )
    )
    (ln

### Training

In [19]:
optimizer = AdamW(dense_model.parameters(), lr=1e-3)
scheduler = StepLR(optimizer, step_size=100, gamma=0.9)
loss_fn = torch.nn.CrossEntropyLoss()
epochs = 2

In [20]:
trainer = Trainer(
    model=dense_model,
    train_data_loader=train_loader,
    test_data_loader=val_loader,
    loss_fn=loss_fn,
    gradient_accumulation_steps=1,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    save_dir="./checkpoints",
    save_every_n=500
)
# Entrenamiento
for epoch in range(epochs):
    avg_train_loss = trainer.train_model_v2(use_amp=True, dtype=torch.bfloat16)
    print(f"Época {epoch+1} - pérdida de entrenamiento: {avg_train_loss:.4f}")

    val_loss = trainer.eval_model()
    print(f"Época {epoch+1} - pérdida de validación: {val_loss:.4f}")

print("Entrenamiento completo.")

  0%|          | 0/11246 [00:00<?, ?it/s]W0403 23:52:31.717000 2672 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warni

Época 1 - pérdida de entrenamiento: 2.1692


val_loss 2.07594: 100%|██████████| 1246/1246 [00:10<00:00, 124.24it/s]


Época 1 - pérdida de validación: 2.0556


loss 2.14952: 100%|██████████| 11246/11246 [03:22<00:00, 55.63it/s]


Época 2 - pérdida de entrenamiento: 2.1181


val_loss 2.07594: 100%|██████████| 1246/1246 [00:10<00:00, 121.79it/s]

Época 2 - pérdida de validación: 2.0556
Entrenamiento completo.


### Testing

In [21]:
greedy_sampling = SamplingParams(greedy=True)

In [22]:
generate_n(
    5,
    prompt="To be",
    model=dense_model,
    device=device,
    config=config,
    params=greedy_sampling,
    encode=encode, decode=decode,
    max_new_tokens=100, use_cache=True,
)

SAMPLE  1/5 
To beat the the the the the the the the the the the the the the the the the the the the the the the the t
********************

SAMPLE  2/5 
To beat the the the the the the the the the the the the the the the the the the the the the the the the t
********************

SAMPLE  3/5 
To beat the the the the the the the the the the the the the the the the the the the the the the the the t
********************

SAMPLE  4/5 
To beat the the the the the the the the the the the the the the the the the the the the the the the the t
********************

SAMPLE  5/5 
To beat the the the the the the the the the the the the the the the the the the the the the the the the t
********************



Por supuesto el greedy sampling funciona mal para un modelo más chico. Esto se debe a que los términos comunes dominan las probabilidades y al ser greedy no hay una componente estocástica que le permita "salir" de esa región de generación.

Pasamos a un esquema estándar (e.g. parámetros de sampleo sugeridos de [Qwen3](https://huggingface.co/Qwen/Qwen3-8B#best-practices) y [GLM5](https://huggingface.co/zai-org/GLM-5#footnote)) de `temperatura` 1, `top_k` "generoso" y `top_p` de 0.9.

In [23]:
standard_sampling = SamplingParams(top_k=20, top_p=0.9)

In [24]:
generate_n(
    5,
    prompt="To be",
    model=dense_model,
    device=device,
    config=config,
    params=standard_sampling,
    encode=encode, decode=decode,
    max_new_tokens=100, use_cache=True,
)

SAMPLE  1/5 
To berime, andes the we mote
That you mat hass les to' derecheare pam paruthe coves.

CORINIUS:
Tooce whe
********************

SAMPLE  2/5 
To bems, he your wede me dor we cory in widend dourearthe nownes at nooike fine cook:
No you car hin ons 
********************

SAMPLE  3/5 
To besome
Wen do the so attoug, thear golens wethe bum our the to the ich presst hius to pre plelveand as
********************

SAMPLE  4/5 
To bett
Whe indsey, sous, shach of they this the now now wirs be, thanl thible nos
To therst wenck thenet
********************

SAMPLE  5/5 
To ben mirsols of ifferore he pon the for sor filuss nove nos
Thims the henconds oble the and sod not but
********************



Los resultados son consistentemente mejores, además de agregar variedad (a diferencia del greedy que es determinístico).

### Ablation: ¿Agrandar el batch size sirve?

Se duplica el batch size y se entrena por un solo epoch para mantener la cantidad de tokens de entrenamiento constante.

In [25]:
from dataclasses import replace

config2 = replace(config, batch_size=2*config.batch_size)

config.batch_size, config2.batch_size

(8, 16)

In [26]:
# dataloaders
num_workers = 0 if device=='mps' else 2

train_loader2 = DataLoader(
    train_dataset,
    batch_size=config2.batch_size,
    shuffle=True,
    drop_last=True,
    pin_memory=True,
    num_workers= num_workers,
)

val_loader2 = DataLoader(
    val_dataset,
    batch_size=config2.batch_size,
    shuffle=False,
    drop_last=True,
    pin_memory=True,
    num_workers=num_workers,
)

In [27]:
dense2 = TinyGPT(config).to(device)
dense_model2 = torch.compile(dense2)

optimizer = AdamW(dense_model2.parameters(), lr=1e-3)
scheduler = StepLR(optimizer, step_size=100, gamma=0.9)
loss_fn = torch.nn.CrossEntropyLoss()
epochs = 1

trainer = Trainer(
    model=dense_model2,
    train_data_loader=train_loader2,
    test_data_loader=val_loader2,
    loss_fn=loss_fn,
    gradient_accumulation_steps=1,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    save_dir="./checkpoints2",
    save_every_n=500
)
# Entrenamiento
for epoch in range(epochs):
    avg_train_loss = trainer.train_model_v2(use_amp=True, dtype=torch.bfloat16)
    print(f"Época {epoch+1} - pérdida de entrenamiento: {avg_train_loss:.4f}")

    val_loss = trainer.eval_model()
    print(f"Época {epoch+1} - pérdida de validación: {val_loss:.4f}")

print("Entrenamiento completo.")

loss 2.05149: 100%|██████████| 5623/5623 [01:39<00:00, 56.74it/s]


Época 1 - pérdida de entrenamiento: 2.0722


val_loss 1.91900: 100%|██████████| 623/623 [00:05<00:00, 115.44it/s]

Época 1 - pérdida de validación: 1.9495
Entrenamiento completo.


Se obtuvo una `valid_loss` menor. Debe observarse que en el caso anterior el primer epoch también tenía una `valid_loss` menor a la del segundo epoch, pero aún así es mayor que la de ahora. Conclusión: agrandar el `batch_size` es útil, incluso a misma cantidad de tokens de entrenamiento.

## MoE model

Usamos directamente la configuración con batch size duplicado.

In [28]:
moe_cfg = MoEArgs(
    num_experts=4,
    num_experts_per_token=2
)

config = GPTConfig(vocab_size=vocab_size, moe=moe_cfg, ff_class=MoEFFN,
                   batch_size=config2.batch_size)

config

GPTConfig(vocab_size=61, block_size=32, batch_size=16, n_embd=64, n_head=4, n_layer=2, dropout=0.1, bias=True, ff_class=<class 'src.mlp.MoEFFN'>, moe=MoEArgs(num_experts=4, num_experts_per_token=2))

In [29]:
moe_m = TinyGPT(config).to(device)
moe_model = torch.compile(moe_m)
moe_model

OptimizedModule(
  (_orig_mod): TinyGPT(
    (token_emb): Embedding(61, 64)
    (pos_emb): Embedding(32, 64)
    (blocks): ModuleList(
      (0-1): 2 x Block(
        (ln1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (ln2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (attn): MultiHeadAttention(
          (heads): ModuleList(
            (0-3): 4 x AttentionHead(
              (key_query_value): Linear(in_features=64, out_features=48, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
          )
          (proj): Linear(in_features=64, out_features=64, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (ff): MoEFFN(
          (moe): MoELayer(
            (experts): ModuleList(
              (0-3): 4 x Expert(
                (net): Sequential(
                  (0): Linear(in_features=64, out_features=128, bias=True)
                  (1): ReLU()
                  (2): Linear(in_features=128, o

In [30]:
# comparamos cantidad de parámetros
print("Dense:", count_trainable_params(dense_model))
print("MoE:",count_trainable_params(moe_model))

Dense: 109952
MoE: 176904


Si bien se mantiene la misma cantidad de parámetros activos, la cantidad total de parámetros aumentó. Se debería esperar:

1. Mayor tiempo de entrenamiento.
2. Tiempo de inferencia un poco más lento (por la pequeña escala), pero con un slowdown menor al de train.
3. Mejor performance.

In [31]:
# replicamos esquema previo, dejamos 2 epochs para ver si este también "overfittea"
optimizer = AdamW(moe_model.parameters(), lr=1e-3)
scheduler = StepLR(optimizer, step_size=100, gamma=0.9)
loss_fn = torch.nn.CrossEntropyLoss()
epochs = 2

In [32]:
dummy_input = torch.tensor([[3,5,44,25],[3,1,2,31],[3,1,2,31]]).to(device)
print("dummy shape",dummy_input.shape)
moe_model(dummy_input)
None

dummy shape torch.Size([3, 4])


In [33]:
trainer = Trainer(
    model=moe_model,
    train_data_loader=train_loader2,
    test_data_loader=val_loader2,
    loss_fn=loss_fn,
    gradient_accumulation_steps=1,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    save_dir="./checkpoints-moe",
    save_every_n=500
)

# Entrenamiento
for epoch in range(epochs):
    avg_train_loss = trainer.train_model_v2(use_amp=True, dtype=torch.bfloat16)
    print(f"Época {epoch+1} - pérdida de entrenamiento: {avg_train_loss:.4f}")

    val_loss = trainer.eval_model()
    print(f"Época {epoch+1} - pérdida de validación: {val_loss:.4f}")

print("Entrenamiento completo.")

  0%|          | 0/5623 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
loss 2.02596: 100%|██████████| 5623/5623 [02:57<00:00, 31.65it/s]


Época 1 - pérdida de entrenamiento: 2.0171


val_loss 1.90756: 100%|██████████| 623/623 [00:08<00:00, 75.04it/s]


Época 1 - pérdida de validación: 1.9373


loss 1.98747: 100%|██████████| 5623/5623 [02:57<00:00, 31.62it/s]


Época 2 - pérdida de entrenamiento: 1.9868


val_loss 1.90479: 100%|██████████| 623/623 [00:07<00:00, 78.79it/s]

Época 2 - pérdida de validación: 1.9366
Entrenamiento completo.


* Se reconfirma el esquema de doble `batch_size` (respecto del default) y saturándose el modelo en 1 epoch, al menos para el tamaño que tienen estos.
* Se validan las hipótesis de entrenamiento más lento (aprox 2x) y slowdown de inferencia (1.6x) menor que el de train.

In [34]:
generate_n(
    5,
    prompt="To be",
    model=moe_model,
    device=device,
    config=config,
    params=standard_sampling,
    encode=encode, decode=decode,
    max_new_tokens=100, use_cache=True,
)

SAMPLE  1/5 
To be dile
O, din the the staingur
Mast the fores hant loter.

SICINIUS:
No, foll the apeath be omed art 
********************

SAMPLE  2/5 
To beat comes, sonot ood.

MENENIUS:
Whely comblenk
Omon al the crotill the com, your thare,
Ince lo wour
********************

SAMPLE  3/5 
To bene, whell whell attes you salont ainst ower paills that, comintens ing deles!

Menath, thee thempor:
********************

SAMPLE  4/5 
To be the that and off you fiell there litul mas.

MENENIUS:
Nor hay to matt thale so that a thim.

BRUTU
********************

SAMPLE  5/5 
To be me, I a perm, they hitis, my
you hensthare dich sust weath all stord the nom o' preat my
An bedalle
********************



El resultado es algo mejor que el del modelo denso. Se observa una mejor capacidad para replicar la estructura del texto de entrenamiento.

## Escalado

Aumentamos el batch size y la hidden dim.

In [35]:
config = GPTConfig(
    vocab_size=vocab_size,
    moe=moe_cfg,
    ff_class=MoEFFN,
    n_embd=1024,
    batch_size=128,
)

config

GPTConfig(vocab_size=61, block_size=32, batch_size=128, n_embd=1024, n_head=4, n_layer=2, dropout=0.1, bias=True, ff_class=<class 'src.mlp.MoEFFN'>, moe=MoEArgs(num_experts=4, num_experts_per_token=2))

In [36]:
# nuevos dataloaders por batch_sz diferente
num_workers = 0 if device=='mps' else 2

train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    drop_last=True,
    pin_memory=True,
    num_workers= num_workers,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    drop_last=True,
    pin_memory=True,
    num_workers=num_workers,
)

In [37]:
moe_m = TinyGPT(config).to(device)
moe_model = torch.compile(moe_m)
moe_model

OptimizedModule(
  (_orig_mod): TinyGPT(
    (token_emb): Embedding(61, 1024)
    (pos_emb): Embedding(32, 1024)
    (blocks): ModuleList(
      (0-1): 2 x Block(
        (ln1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (ln2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (attn): MultiHeadAttention(
          (heads): ModuleList(
            (0-3): 4 x AttentionHead(
              (key_query_value): Linear(in_features=1024, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
          )
          (proj): Linear(in_features=1024, out_features=1024, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (ff): MoEFFN(
          (moe): MoELayer(
            (experts): ModuleList(
              (0-3): 4 x Expert(
                (net): Sequential(
                  (0): Linear(in_features=1024, out_features=2048, bias=True)
                  (1): ReLU()
                  (2): Linear(

In [38]:
# replicamos esquema previo + 1 solo epoch
optimizer = AdamW(moe_model.parameters(), lr=3e-4) # karpathy number
scheduler = StepLR(optimizer, step_size=100, gamma=0.9)
loss_fn = torch.nn.CrossEntropyLoss()
epochs = 1

In [39]:
trainer = Trainer(
    model=moe_model,
    train_data_loader=train_loader,
    test_data_loader=val_loader,
    loss_fn=loss_fn,
    gradient_accumulation_steps=1,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    save_dir="./checkpoints-moe2",
    save_every_n=500
)

# Entrenamiento
for epoch in range(epochs):
    avg_train_loss = trainer.train_model_v2(use_amp=True, dtype=torch.bfloat16)
    print(f"Época {epoch+1} - pérdida de entrenamiento: {avg_train_loss:.4f}")

    val_loss = trainer.eval_model()
    print(f"Época {epoch+1} - pérdida de validación: {val_loss:.4f}")

print("Entrenamiento completo.")

loss 1.36308: 100%|██████████| 702/702 [03:13<00:00,  3.63it/s]


Época 1 - pérdida de entrenamiento: 1.3579


val_loss 1.38036: 100%|██████████| 77/77 [00:06<00:00, 12.79it/s]

Época 1 - pérdida de validación: 1.5680
Entrenamiento completo.


In [40]:
# vemos greedy
generate_n(
    5,
    prompt="To be",
    model=moe_model,
    device=device,
    config=config,
    params=greedy_sampling,
    encode=encode, decode=decode,
    max_new_tokens=100, use_cache=True,
)

SAMPLE  1/5 
To be consul, and the common to the common the common the common the common the common the common the com
********************

SAMPLE  2/5 
To be consul, and the common to the common the common the common the common the common the common the com
********************

SAMPLE  3/5 
To be consul, and the common to the common the common the common the common the common the common the com
********************

SAMPLE  4/5 
To be consul, and the common to the common the common the common the common the common the common the com
********************

SAMPLE  5/5 
To be consul, and the common to the common the common the common the common the common the common the com
********************



Como ya se observó, el decoding greedy sufre del "latching" a términos muy comunes que dominan.

In [41]:
# ahora el standard
generate_n(
    5,
    prompt="To be",
    model=moe_model,
    device=device,
    config=config,
    params=standard_sampling,
    encode=encode, decode=decode,
    max_new_tokens=100, use_cache=True,
)

SAMPLE  1/5 
To beggar the say have patricians in them.

COMINIUS:
They senators now and to stay
this charge it, but n
********************

SAMPLE  2/5 
To being out your and in this from he
comper me of him of this graves.

COMINIUS:
Not what every begn blo
********************

SAMPLE  3/5 
To beard, content;
and you present their stribunes came,
And what decling farmint
The saw forth and grati
********************

SAMPLE  4/5 
To be consul.

SICINIUS:
Sir, I can he country: the loss,
Which see this present
The pretty clamount held
********************

SAMPLE  5/5 
To be of the common of my say with were infort.

VIRGILIA:
Let come are fight
The bown and clours, and he
********************



Se observa que al escalar el modelo, el texto generado se vuelve mucho más coherente. Dado que sigue siendo una escala chica, los tiempos de entrenamiento e inferencia casi no se ven modificados.

## Tokenizer preentrenado

Se utiliza el tokenizer del modelo [GPT 2](https://huggingface.co/openai-community/gpt2).

Debe observarse sin embargo, que en modelos tan chicos utilizar un tokenizer con un vocabulario más grande requiere una capa de embedding inicial más grande, lo cual a su vez aumenta los parámetros del modelo. No son parámetros de "inteligencia", pero sí de "conocimiento".

In [42]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "openai-community/gpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Vocab size previo:", vocab_size)
print(f"Vocab size {MODEL_NAME}:", len(tokenizer))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Vocab size previo: 61
Vocab size openai-community/gpt2: 50257


In [43]:
def encode(s):
    return tokenizer(s)['input_ids']
def decode(l):
    return tokenizer.decode(l)

# mostramos uso
ss = encode(text[:30])
print(ss)

dd = decode(ss)
print(dd)

[5962, 22307, 25, 198, 8421, 356, 386, 344]
First Citizen:
Before we proce


In [44]:
# 0.9-train/0.1-val split
data = torch.tensor(encode(text), dtype=torch.long)
split = int(0.9 * len(data))
train_data = data[:split]
val_data = data[split:]

train_data.shape, val_data.shape

Token indices sequence length is longer than the specified maximum sequence length for this model (30645 > 1024). Running this sequence through the model will result in indexing errors


(torch.Size([27580]), torch.Size([3065]))

In [45]:
# nuevo config por nuevo tamaño de vocabulario
config_gpt2 = replace(config, vocab_size=len(tokenizer))

In [46]:
# replicamos el mismo dataset utilizando tokens ahora
train_dataset = CharDataset(train_data, config_gpt2.block_size)
val_dataset = CharDataset(val_data, config_gpt2.block_size)

In [47]:
# nuevos dataloaders por data diferente
num_workers = 0 if device=='mps' else 2

train_loader = DataLoader(
    train_dataset,
    batch_size=config_gpt2.batch_size,
    shuffle=True,
    drop_last=True,
    pin_memory=True,
    num_workers= num_workers,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config_gpt2.batch_size,
    shuffle=False,
    drop_last=True,
    pin_memory=True,
    num_workers=num_workers,
)

In [48]:
# rehacemos el modelo escalado
moe_m = TinyGPT(config_gpt2).to(device)
moe_model = torch.compile(moe_m)
moe_model

OptimizedModule(
  (_orig_mod): TinyGPT(
    (token_emb): Embedding(50257, 1024)
    (pos_emb): Embedding(32, 1024)
    (blocks): ModuleList(
      (0-1): 2 x Block(
        (ln1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (ln2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (attn): MultiHeadAttention(
          (heads): ModuleList(
            (0-3): 4 x AttentionHead(
              (key_query_value): Linear(in_features=1024, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
          )
          (proj): Linear(in_features=1024, out_features=1024, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (ff): MoEFFN(
          (moe): MoELayer(
            (experts): ModuleList(
              (0-3): 4 x Expert(
                (net): Sequential(
                  (0): Linear(in_features=1024, out_features=2048, bias=True)
                  (1): ReLU()
                  (2): Line

In [49]:
optimizer = AdamW(moe_model.parameters(), lr=3e-4) # karpathy number
scheduler = StepLR(optimizer, step_size=100, gamma=0.9)
loss_fn = torch.nn.CrossEntropyLoss()
epochs = 1

In [50]:
trainer = Trainer(
    model=moe_model,
    train_data_loader=train_loader,
    test_data_loader=val_loader,
    loss_fn=loss_fn,
    gradient_accumulation_steps=1,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    save_dir="./checkpoints-moe-gpt2",
    save_every_n=500
)

# Entrenamiento
for epoch in range(epochs):
    avg_train_loss = trainer.train_model_v2(use_amp=True, dtype=torch.bfloat16)
    print(f"Época {epoch+1} - pérdida de entrenamiento: {avg_train_loss:.4f}")

    val_loss = trainer.eval_model()
    print(f"Época {epoch+1} - pérdida de validación: {val_loss:.4f}")

print("Entrenamiento completo.")

loss 1.83134: 100%|██████████| 215/215 [03:18<00:00,  1.08it/s]


Época 1 - pérdida de entrenamiento: 1.7728


val_loss 3.66008: 100%|██████████| 23/23 [00:04<00:00,  5.27it/s]

Época 1 - pérdida de validación: 5.2174
Entrenamiento completo.


In [51]:
# ahora el standard
generate_n(
    5,
    prompt="To be",
    model=moe_model,
    device=device,
    config=config,
    params=standard_sampling,
    encode=encode, decode=decode,
    max_new_tokens=100, use_cache=True,
)

SAMPLE  1/5 
To be content: if you have been my
With them as they love.

MENENIUS:
What is it? to the custom?

First Senator:
They have much corn; you have been a
With what then?

Messenger:
He that, come, that you couldst thou have been
Were he met your voices: adieu.

First Citizen:
No, sir, what stock he was anon,
We are to do
********************

SAMPLE  2/5 
To be abhorr'd a single
man. We'll potch at, you are not
With modest warrant.

BRUTUS:
He's a brace:
He's coming.

First Citizen:
He's not flatter us there.

BRUTUS:
How now, if they you not,
Though Marcius shall be consul:
The gods forbid!
You so remain.

MENENIUS:
Pray you, what
********************

SAMPLE  3/5 
To be sworn by, and presume to give forth
To hear hither your voices? What would you have power'd
As you were born in ourselves to do; where being three oppos,
You, your opinion, the belly answer?

BRUTUS:
He's coming.

MARCIUS:
The gods give you not cry you!

SICINIUS:
So, how comes't and I will practise
My fortun

* Los textos generados son mucho más largos, producto de que la cantidad de caracteres por token aumento a más del triple en promedio.
* Para un modelo del mismo tamaño (en parámetros no-embedding), al incorporar tokens de sub-palabra el nivel de consistencia aumentó notablemente.
* La loss aumentó considerablemente ya que es mucho más difícil predecir la siguiente subpalabra sobre un pool ~3 órdenes de magnitud mayor.

## Conclusiones

* El MoE es un esquema muy útil para inducir _sparsity_ en los subloques MLP de los gpt-like language models, manteniendo una velocidad competitiva con modelos densos de la misma cantidad de parámetros activos pero con un conocimiento mayor.
* A escalas chicas es difícil mostrar el speedup propio de un MoE ya que los tamaños de las matrices de los modelos densos no son suficientemente grandes para convertirse en el cuello de botella de cómputo. Se decidió no hacer entrenamientos grandes por disponibilidad de recursos (colab gratuito).
* Al escalar los modelos se observa rápidamente cómo los mismos adquieren mayor coherencia, incluso para un tokenizer a nivel de caracteres.
* A misma cantidad de parámetros, incrementar la "complejidad" de los tokens de caracter a sub-palabra mejora enormemente la coherencia del modelo. Sin embargo, es notable que se trabajó sobre un dataset que es un subconjunto muy pequeño de la masa necesaria para poder entrenar todos los embeddings correctamente. Tokenización más compleja -> mayor necesidad de datos pero mayor expresividad -> mayor coherencia sobre tokens "observados".